# Graphiko: Business Cluster Channel Descriptions

Connects to MongoDB, finds the latest **business** cluster in the `finder` DB context,
and prints channel descriptions for channels in that cluster.


In [1]:
# Install dependencies (Colab-friendly)
!pip install -q pymongo dnspython pinecone


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 32.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 21.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 742.8/742.8 kB 36.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 280.7/280.7 kB 17.6 MB/s eta 0:00:00


In [2]:
from pymongo import MongoClient
from google.colab import userdata

try:
    uri = userdata.get('MONGODB_URI')
    client = MongoClient(uri)
    client.admin.command('ping')
    print('Pinged your deployment. You successfully connected to MongoDB!')
except Exception as e:
    print(f'An error occurred while connecting: {e}')
    raise


Pinged your deployment. You successfully connected to MongoDB!


In [3]:
from pinecone import Pinecone

# Pinecone connection (secret key from Colab secrets)
try:
    pinecone_api_key = userdata.get('PINECONE_API_KEY')
    pc = Pinecone(api_key=pinecone_api_key)
    pinecone_index = pc.Index('finder')
    pinecone_namespace = 'ChannelDescriptions'
    print('Connected to Pinecone index: finder (namespace: ChannelDescriptions)')
except Exception as e:
    print(f'An error occurred while connecting to Pinecone: {e}')
    raise


Connected to Pinecone index: finder (namespace: ChannelDescriptions)


In [4]:
# Finder DB collections
# Reusing the same cluster lookup approach as the existing Grafiko notebook.
db = client['finder']
clusters_col = db['ChannelDescriptions_clusters']
items_col = db['ChannelDescriptions_items']
channels_col = db['channels']

# 1) Latest cluster version
latest = clusters_col.find_one(sort=[('version', -1), ('createdAt', -1)])
latest_version = latest['version'] if latest else None
print(f'Latest cluster version: {latest_version}')

# 2) Business cluster in latest version
business_cluster_id = None
if latest_version is None:
    print('No clusters found.')
else:
    business_cluster = clusters_col.find_one({
        'version': latest_version,
        'name': {'$regex': '^business', '$options': 'i'}
    })
    if business_cluster:
        business_cluster_id = business_cluster['_id']
        print(f"Business cluster id: {business_cluster_id}")
        print(f"Business cluster name: {business_cluster.get('name')}")
    else:
        print('No business cluster found in latest version.')


Latest cluster version: 2
Business cluster id: 69e41878685f30ed081ec5a6
Business cluster name: Business, Venture Capital, and Entrepreneurship


In [5]:
# 3) Get channel descriptions for channels in the business cluster and print them
if business_cluster_id is None:
    print('Cannot fetch channel descriptions without a business cluster id.')
else:
    # Channel IDs mapped to this cluster
    item_docs = list(items_col.find(
        {'clusterId': business_cluster_id},
        {'_id': 0, 'textId': 1}
    ))
    channel_ids = [d['textId'] for d in item_docs if d.get('textId')]

    # Fetch channel metadata + descriptions
    channel_docs = list(channels_col.find(
        {'channelId': {'$in': channel_ids}},
        {'_id': 0, 'channelId': 1, 'title': 1, 'description': 1}
    ))

    print(f'Channels in business cluster: {len(channel_docs)}')
    print('=' * 100)
    for i, ch in enumerate(channel_docs, start=1):
        title = ch.get('title') or 'Unknown title'
        desc = ch.get('description') or '<no description>'
        print(f"{i}. {title} ({ch.get('channelId')})")
        print(desc)
        print('-' * 100)


Channels in business cluster: 14
1. Bg2 Pod (UC-yRDvpR99LUc5l7i7jLzew)
Open Source bi-weekly convo w @altcap & @bgurley on all things tech, markets, investing & capitalism

----------------------------------------------------------------------------------------------------
2. Asianometry (UC1LpsuAUaKoMzzJSEt5WImw)
Video essays on business, economics, and history. Sometimes about Asia, but not always. 

For general mail: hello@asianometry.com
For business inquiries: business@asianometry.com

I don't have an Asianometry Telegram. Don't fall for comment scams on YouTube, please. 

----------------------------------------------------------------------------------------------------
3. Lenny's Podcast (UC6t1O76G0jYXOAoYCm153dA)
Interviews with world-class product leaders and growth experts to uncover concrete, actionable, and tactical advice to help you build, launch, and grow your own product.
--------------------------------------------------------------------------------------------------

In [6]:
def get_or_create_channel_description_embeddings(
    channel_docs,
    index,
    namespace='ChannelDescriptions',
    model='multilingual-e5-large',
    batch_size=96
):
    """
    Returns a mapping of channelId -> embedding for the provided channel docs.
    Steps:
      1) Attempt to fetch existing vectors from Pinecone by channelId.
      2) For any missing ids, embed descriptions with multilingual-e5-large and upsert.
    """
    channel_map = {
        str(d.get('channelId')): d
        for d in channel_docs
        if d.get('channelId')
    }
    channel_ids = list(channel_map.keys())
    if not channel_ids:
        return {}

    embeddings_by_channel = {}

    # 1) Fetch existing vectors
    for i in range(0, len(channel_ids), batch_size):
        batch_ids = channel_ids[i:i + batch_size]
        fetch_response = index.fetch(ids=batch_ids, namespace=namespace)
        vectors = fetch_response.get('vectors', {}) if isinstance(fetch_response, dict) else fetch_response.vectors
        for cid, vector_data in vectors.items():
            if isinstance(vector_data, dict):
                embeddings_by_channel[cid] = vector_data.get('values')
            else:
                embeddings_by_channel[cid] = getattr(vector_data, 'values', None)

    missing_ids = [cid for cid in channel_ids if cid not in embeddings_by_channel]

    # 2) Embed + upsert missing descriptions
    if missing_ids:
        vectors_to_upsert = []
        descriptions = [channel_map[cid].get('description') or '' for cid in missing_ids]

        for i in range(0, len(missing_ids), batch_size):
            ids_batch = missing_ids[i:i + batch_size]
            texts_batch = descriptions[i:i + batch_size]

            embedding_result = pc.inference.embed(
                model=model,
                inputs=texts_batch,
                parameters={'input_type': 'passage', 'truncate': 'END'}
            )

            # Works across Pinecone response object variants
            rows = getattr(embedding_result, 'data', embedding_result)
            for cid, ch_doc, row in zip(ids_batch, [channel_map[x] for x in ids_batch], rows):
                values = row.get('values') if isinstance(row, dict) else getattr(row, 'values', None)
                if values is None:
                    values = row.get('embedding') if isinstance(row, dict) else getattr(row, 'embedding', None)

                metadata = {
                    'title': ch_doc.get('title') or '',
                    'description': ch_doc.get('description') or ''
                }
                vectors_to_upsert.append({'id': cid, 'values': values, 'metadata': metadata})
                embeddings_by_channel[cid] = values

        if vectors_to_upsert:
            index.upsert(vectors=vectors_to_upsert, namespace=namespace)

    return embeddings_by_channel


In [7]:
# 4) Retrieve-or-create embeddings for the business cluster channel descriptions
if business_cluster_id is None:
    cluster_channel_description_embeddings = {}
else:
    cluster_channel_description_embeddings = get_or_create_channel_description_embeddings(
        channel_docs=channel_docs,
        index=pinecone_index,
        namespace=pinecone_namespace,
        model='multilingual-e5-large'
    )

print(f'Embeddings returned: {len(cluster_channel_description_embeddings)}')
cluster_channel_description_embeddings


Embeddings returned: 14


{'UCznv7Vf9nBdJYvBagFdAHWw': [0.0216369629,
  -0.00458526611,
  -0.0304718018,
  -0.028427124,
  0.0259399414,
  0.000227570534,
  -0.0076713562,
  0.0665893555,
  0.0492553711,
  -0.012840271,
  0.0323791504,
  0.0243530273,
  -0.0199127197,
  -0.0514831543,
  0.000400543213,
  -0.037689209,
  -0.0393676758,
  0.0156555176,
  0.0044593811,
  -0.0342712402,
  0.0584411621,
  -0.0133285522,
  -0.0343933105,
  -0.034942627,
  -0.0106506348,
  -0.0158081055,
  -0.00799560547,
  -0.0269775391,
  -0.0145263672,
  -0.0222167969,
  -0.0175628662,
  0.008644104,
  -0.00979614258,
  -0.0558776855,
  -0.024520874,
  0.0285949707,
  0.0277557373,
  0.0584411621,
  -0.0540771484,
  0.0304718018,
  -0.00427627563,
  0.0325317383,
  -0.0458984375,
  -0.0444946289,
  0.0362243652,
  0.0283660889,
  0.0118637085,
  0.01902771,
  -0.0182342529,
  0.0176086426,
  0.0193481445,
  -0.0268707275,
  -0.00656890869,
  -0.0167541504,
  -0.0195922852,
  0.0109863281,
  -0.0300750732,
  0.0170135498,
  -0.05770